# 04 · The FastAPI service, the run log and the evaluation

**What you will learn**
1. Call every endpoint in-process with FastAPI's `TestClient` (same code as `uvicorn pharma_agent.api:app`)
2. Read the ACID run log in SQLite
3. How the evaluation scores the agent, with a small live run and the latest full report

In [ ]:
# Setup: make the project importable and quiet the HTTP logs
import sys, logging, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
logging.getLogger("httpx").setLevel(logging.WARNING)

def show(obj, limit=1500):
    """Pretty-print a pydantic model, dict or list, cut to `limit` characters."""
    if hasattr(obj, "model_dump"):
        obj = obj.model_dump()
    if isinstance(obj, list):
        obj = [o.model_dump() if hasattr(o, "model_dump") else o for o in obj]
    text = json.dumps(obj, indent=2, ensure_ascii=False, default=str)
    print(text if len(text) <= limit else text[:limit] + "\n... (cut)")

In [ ]:
from fastapi.testclient import TestClient
from pharma_agent.api import create_app
from pharma_agent.config import load_config
from pharma_agent.service import PharmaAgent
from pharma_agent.store import Store

cfg = load_config(overrides={"store.path": ":memory:"})
store = Store(":memory:")

async def factory():
    return await PharmaAgent.create(cfg, store=store)

client = TestClient(create_app(factory))
client.__enter__()          # runs the app's startup (builds the agent once)
print(client.get("/health").json())

In [ ]:
for t in client.get("/tools").json():
    print(f"{t['name']:26s} {list(t['args'])}")
print()
for a in client.get("/agents").json():
    print(f"{a['name']:17s} {a['tools']}")

## `POST /ask`: the structured, cited answer

In [ ]:
body = client.post("/ask", json={"question": "What are the contraindications for apixaban?"}).json()
print(body["answer"]["answer"], "\n")
for p in body["answer"]["key_points"]:
    print("-", p["text"], p["citations"])
print("\nGuardrail:", body["guardrail"], "| latency:", body["latency_ms"], "ms")

In [ ]:
print("Validation:", client.post("/ask", json={"question": "hi"}).status_code, "(422 = rejected, too short)")
print(client.get("/runs").json())

## The run log is ACID
`save_run` writes the run and all its evidence rows in **one transaction**: both land, or neither does.
The tests (`tests/test_store_acid.py`) force a failure halfway through and check nothing is left behind.

In [ ]:
run = store.get_run(body["run_id"])
print(run["question"], "|", len(run["evidence"]), "evidence rows |", run["created_at"])
print("runs:", store.count("runs"), " run_evidence:", store.count("run_evidence"))

## Evaluation
Each question in `eval/questions.jsonl` lists the agents and sources it should need. The harness measures:

| Metric | Meaning |
|---|---|
| routing_exact | supervisor chose exactly the expected agents |
| source_recall | expected sources returned evidence |
| citation_precision | cited ids that were real evidence (before the guardrail) |
| grounded_points | key points with at least one valid citation |
| term_hit | expected key terms appear in the answer |

A small live run (2 questions):

In [ ]:
from pharma_agent.evaluation import load_questions, run_eval
questions = load_questions(ROOT / "eval" / "questions.jsonl")
agent = client.app.state.agent
mini = await run_eval(agent, [questions[0], questions[12]], delay_seconds=10)
show(mini["summary"])

And the latest full run, saved by `python -m pharma_agent eval`:

In [ ]:
latest = sorted((ROOT / "eval" / "results").glob("eval_*.json"))[-1]
report = json.loads(latest.read_text(encoding="utf-8"))
print(latest.name)
show(report["summary"])
print()
for r in report["rows"]:
    s = r.get("scores", {})
    print(f"{r['id']}  routing={s.get('routing_exact', '-')}  sources={s.get('source_recall', '-')}  "
          f"cit_prec={s.get('citation_precision', '-')}  terms={s.get('term_hit', '-')}  {r.get('error', '')[:60]}")

In [ ]:
client.__exit__(None, None, None)